# Verify HILDA GeoTIFF vs national CSV

The dashboard reads **the same** classified pixels from `rasters/hilda/geotiff/hilda_YYYY.tif` as are summarized in `outputs/hilda_lithuania_timeseries.csv`. If the **map** looks almost all agriculture but the **donut** shows ~35% forest, usual causes are:

- **Stale browser cache** (old GeoTIFF while CSV was re-exported). The dashboard now fetches GeoTIFF with `cache: no-store` and does not keep an in-memory cache.
- **Visual bias**: agriculture yellow is bright; forest green is darker — the eye overestimates yellow area.

This notebook **counts pixels per class** in a GeoTIFF and compares to the CSV for the same year. They must match if export completed successfully.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import rasterio
except ImportError as e:
    raise SystemExit("Install rasterio in this kernel: pip install rasterio") from e


def find_data_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "lt_subbasins.json").is_file():
            return d
    raise FileNotFoundError("Set DATA_ROOT to your Data repo (folder with lt_subbasins.json).")


DATA_ROOT = find_data_root(Path.cwd())
CSV_PATH = DATA_ROOT / "outputs" / "hilda_lithuania_timeseries.csv"
GEOTIFF_DIR = DATA_ROOT / "rasters" / "hilda" / "geotiff"

print("DATA_ROOT:", DATA_ROOT)
print("CSV exists:", CSV_PATH.is_file())
print("GeoTIFF dir:", GEOTIFF_DIR.is_dir())

In [ ]:
CLASS_IDS = (1, 2, 3, 4, 5)
ID_TO_NAME = {1: "Water", 2: "Wetland", 3: "Urban", 4: "Agriculture", 5: "Forest"}


def raster_class_counts(tif_path: Path) -> dict[int, int]:
    with rasterio.open(tif_path) as src:
        d = src.read(1)
    out = {}
    for v in CLASS_IDS:
        n = int((d == v).sum())
        if n:
            out[v] = n
    return out


def csv_counts_for_year(df: pd.DataFrame, year: int) -> dict[int, int]:
    sub = df.loc[df["year"] == year, ["class_id", "count"]]
    return {int(r["class_id"]): int(r["count"]) for _, r in sub.iterrows()}


def compare_year(year: int, df: pd.DataFrame) -> None:
    tif = GEOTIFF_DIR / f"hilda_{year}.tif"
    if not tif.is_file():
        print(f"{year}: missing {tif.name}")
        return
    r_c = raster_class_counts(tif)
    c_c = csv_counts_for_year(df, year)
    r_total = sum(r_c.values())
    c_total = sum(c_c.values())
    all_ids = sorted(set(r_c) | set(c_c))
    ok = r_c == c_c and r_total == c_total
    print(f"{year}: raster total={r_total} CSV total={c_total} {'OK' if ok else 'MISMATCH'}")
    for vid in all_ids:
        rn = r_c.get(vid, 0)
        cn = c_c.get(vid, 0)
        label = ID_TO_NAME.get(vid, str(vid))
        if rn != cn:
            print(f"  {label} ({vid}): raster={rn} csv={cn}")
    if ok:
        for vid in all_ids:
            label = ID_TO_NAME.get(vid, str(vid))
            pct = 100.0 * r_c[vid] / r_total if r_total else 0
            print(f"  {label}: {pct:.2f}%")

In [ ]:
df = pd.read_csv(CSV_PATH)

# Change to any year you use on the map slider
SAMPLE_YEAR = 1990
compare_year(SAMPLE_YEAR, df)

In [ ]:
# Optional: scan all years (slow). Set RUN_ALL = True to run.
RUN_ALL = False

if RUN_ALL:
    years = sorted(df["year"].unique())
    bad = []
    for y in years:
        tif = GEOTIFF_DIR / f"hilda_{int(y)}.tif"
        if not tif.is_file():
            bad.append((y, "missing tif"))
            continue
        r_c = raster_class_counts(tif)
        c_c = csv_counts_for_year(df, int(y))
        if r_c != c_c or sum(r_c.values()) != sum(c_c.values()):
            bad.append((y, "counts differ"))
    print("Years checked:", len(years))
    print("Problems:", bad if bad else "none")